# Evaluate Fine-Tuned LLM in RAG Pipeline

This notebook evaluates the fine-tuned Mistral QLoRA adapter in the Turkish legal RAG pipeline.

Comparison:
- Base Mistral-7B-Instruct-v0.2 RAG
- Fine-tuned Mistral QLoRA RAG

The retrieval contexts are kept the same for both models to isolate generation quality.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime > Change runtime type > T4 GPU seç.")

CUDA available: True
GPU: Tesla T4


In [3]:
import os

project_path = "/content/drive/MyDrive/turkish_legal_rag"

processed_path = f"{project_path}/data/processed"
outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"
faiss_path = f"{outputs_path}/faiss"

adapter_path = f"{models_path}/mistral_legal_qlora_adapter_300steps"

retrieval_corpus_path = f"{processed_path}/retrieval_corpus.csv"
test_qa_path = f"{processed_path}/test_qa.csv"
faiss_index_path = f"{faiss_path}/baseline_faiss.index"

print("Project exists:", os.path.exists(project_path))
print("Retrieval corpus exists:", os.path.exists(retrieval_corpus_path))
print("Test QA exists:", os.path.exists(test_qa_path))
print("FAISS index exists:", os.path.exists(faiss_index_path))
print("Adapter exists:", os.path.exists(adapter_path))
print("Adapter model exists:", os.path.exists(f"{adapter_path}/adapter_model.safetensors"))
print("Adapter config exists:", os.path.exists(f"{adapter_path}/adapter_config.json"))

Project exists: True
Retrieval corpus exists: True
Test QA exists: True
FAISS index exists: True
Adapter exists: True
Adapter model exists: True
Adapter config exists: True


In [4]:
!pip install -q -U transformers accelerate peft bitsandbytes sentence-transformers faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 99.9 MB/s eta 0:00:00


In [5]:
import os
import re
import gc
import json
import numpy as np
import pandas as pd
import torch
import faiss

from tqdm import tqdm

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

In [6]:
chunks_df = pd.read_csv(retrieval_corpus_path)
test_qa_df = pd.read_csv(test_qa_path)

print("Chunks:", chunks_df.shape)
print("Test QA:", test_qa_df.shape)

display(chunks_df.head())
display(test_qa_df.head())

Chunks: (3775, 5)
Test QA: (1500, 2)


,chunk_id,source_context_id,source,chunk_text,chunk_len
0,chunk_000000,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,263
1,chunk_000001,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Dünya milletleri ailesinin eşit haklara sahip ...,194
2,chunk_000002,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Millet iradesinin mutlak üstünlüğü, egemenliği...",276
3,chunk_000003,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Kuvvetler ayrımının, Devlet organları arasında...",256
4,chunk_000004,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Hiçbir faaliyetin Türk milli menfaatlerinin, T...",370


,question,answer
0,Anayasanın 90. Maddesi Nasıl Uygulanır?,Milletlerarası antlaşmaların TBMM tarafından o...
1,Hukukta 'legitimate expectation' nedir?,"Legitimate expectation, bir kişinin belirli bi..."
2,"Anayasa madde 172'ye göre, devletin sanayi ve ...","Anayasa madde 172'ye göre, devlet, sanayi ve t..."
3,"Anayasa madde 158, uyuşmazlık mahkemesi'nin ku...","Anayasa madde 158'e göre, uyuşmazlık mahkemesi..."
4,"Bir grup avukat, Türkiye Büyük Millet Meclisi ...","Anayasanın 94. Maddesi, Türkiye Büyük Millet M..."


In [7]:
eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print("Eval sample:", eval_df.shape)
display(eval_df.head())

Eval sample: (20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [8]:
def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    return [t for t in tokens if t not in stopwords and len(t) > 1]


def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())

In [9]:
def detect_source_filter(query):
    q = str(query).lower()

    if "anayasa" in q or "cumhurbaşkanı" in q or "tbmm" in q or "milletvekili" in q:
        return "Türkiye Cumhuriyeti Anayasası"

    if "medeni kanun" in q or "miras" in q or "evlen" in q or "boşan" in q or "vesayet" in q:
        return "Türk Medeni Kanunu"

    if "borçlar kanunu" in q or "sözleşme" in q or "kira" in q or "kiracı" in q:
        return "Türk Borçlar Kanunu"

    if "ceza muhakemesi" in q or "cmk" in q or "gözaltı" in q or "tutuklama" in q or "yakalama" in q:
        return "Ceza Muhakemesi Kanunu"

    if "türk ceza kanunu" in q or "tck" in q or "suç" in q or "hapis" in q or "sanık" in q:
        return "Türk Ceza Kanunu"

    if "iş kanunu" in q or "işçi" in q or "işveren" in q:
        return "Türkiye Cumhuriyeti İş Kanunu"

    if "bilgi edinme" in q:
        return "Bilgi Edinme Kanunu"

    if "bayrak" in q:
        return "Türk Bayrağı Tüzüğü"

    return None


def extract_article_numbers(query):
    q = str(query).lower()

    patterns = [
        r"madde\s*(\d+)",
        r"maddesi\s*(\d+)",
        r"(\d+)\.\s*madde",
        r"(\d+)\s*inci\s*madde",
        r"(\d+)\s*ıncı\s*madde",
        r"(\d+)\s*uncu\s*madde",
        r"(\d+)\s*üncü\s*madde"
    ]

    found = []

    for pattern in patterns:
        found.extend(re.findall(pattern, q))

    return list(set(found))


def article_bonus_score(chunk_text, article_numbers):
    text = str(chunk_text).lower()

    for article_no in article_numbers:
        if f"madde {article_no}" in text:
            return 1.0
        if f"madde {article_no}-" in text:
            return 1.0
        if f"madde {article_no} –" in text:
            return 1.0
        if f"madde {article_no}." in text:
            return 1.0

    return 0.0

In [10]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(
    embedding_model_name,
    device="cpu"
)

if os.path.exists(faiss_index_path):
    index = faiss.read_index(faiss_index_path)
    print("Loaded existing FAISS index:", faiss_index_path)
else:
    print("FAISS index not found. Building temporary index...")
    corpus_embeddings = embedding_model.encode(
        chunks_df["chunk_text"].astype(str).tolist(),
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype("float32")

    faiss.normalize_L2(corpus_embeddings)

    index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
    index.add(corpus_embeddings)

print("FAISS vectors:", index.ntotal)

tokenized_corpus = [
    simple_turkish_tokenize(text)
    for text in chunks_df["chunk_text"].astype(str).tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded existing FAISS index: /content/drive/MyDrive/turkish_legal_rag/outputs/faiss/baseline_faiss.index
FAISS vectors: 3775
BM25 ready.


In [11]:
base_reranker_model_name = "seroe/bge-reranker-v2-m3-turkish-triplet"

reranker = CrossEncoder(
    base_reranker_model_name,
    device="cpu",
    max_length=512
)

print("Reranker loaded:", base_reranker_model_name)

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reranker loaded: seroe/bge-reranker-v2-m3-turkish-triplet


In [13]:
def retrieve_contexts_for_generation(
    query,
    candidate_k=20,
    context_k=3,
    alpha=0.5,
    article_weight=0.15
):
    source_filter = detect_source_filter(query)
    article_numbers = extract_article_numbers(query)

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(
        query_embedding,
        len(chunks_df)
    )

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(chunks_df))
    ])

    bm25_scores = np.array(
        bm25.get_scores(simple_turkish_tokenize(query))
    )

    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    hybrid_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    article_bonus = np.array([
        article_bonus_score(text, article_numbers)
        for text in chunks_df["chunk_text"].astype(str).tolist()
    ])

    final_scores = hybrid_scores + article_weight * article_bonus

    candidate_indices = np.argsort(final_scores)[::-1]

    # Optional source-aware filter
    filtered_indices = []

    if source_filter is not None:
        for idx in candidate_indices:
            source = str(chunks_df.iloc[idx]["source"]).lower()
            if source == source_filter.lower():
                filtered_indices.append(idx)

        # Eğer filtre çok az sonuç verdiyse fallback yap
        if len(filtered_indices) < context_k:
            filtered_indices = list(candidate_indices)

    else:
        filtered_indices = list(candidate_indices)

    top_candidate_indices = filtered_indices[:candidate_k]

    candidates = []

    for original_rank, idx in enumerate(top_candidate_indices, start=1):
        candidates.append({
            "original_rank": original_rank,
            "chunk_id": str(chunks_df.iloc[idx]["chunk_id"]),
            "source": str(chunks_df.iloc[idx]["source"]),
            "score": float(final_scores[idx]),
            "chunk_text": str(chunks_df.iloc[idx]["chunk_text"])
        })

    # Rerank candidates
    pairs = [
        [query, item["chunk_text"]]
        for item in candidates
    ]

    rerank_scores = reranker.predict(
        pairs,
        batch_size=8,
        show_progress_bar=False
    )

    for item, rerank_score in zip(candidates, rerank_scores):
        item["rerank_score"] = float(rerank_score)

    reranked_candidates = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    final_results = reranked_candidates[:context_k]

    return final_results

In [14]:
retrieval_rows = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    question = row["question"]
    expected_answer = row["answer"]

    retrieved = retrieve_contexts_for_generation(
        question,
        candidate_k=20,
        context_k=3,
        alpha=0.5
    )

    retrieval_rows.append({
        "index": i,
        "question": question,
        "expected_answer": expected_answer,
        "retrieved_contexts": [r["chunk_text"] for r in retrieved],
        "retrieved_chunk_ids": [r["chunk_id"] for r in retrieved],
        "retrieved_sources": [r["source"] for r in retrieved]
    })

retrieval_eval_df = pd.DataFrame(retrieval_rows)

display(retrieval_eval_df.head())

100%|██████████| 20/20 [05:19<00:00, 15.99s/it]


,index,question,expected_answer,retrieved_contexts,retrieved_chunk_ids,retrieved_sources
0,0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,[Cumhurbaşkanlığı seçiminde birinci oylamada g...,"[chunk_000537, chunk_000277, chunk_000366]","[Türkiye Cumhuriyeti Anayasası, Türkiye Cumhur..."
1,1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","[Madde 10 – Herkes, dil, ırk, renk, cinsiyet, ...","[chunk_000274, chunk_000277, chunk_000366]","[Türkiye Cumhuriyeti Anayasası, Türkiye Cumhur..."
2,2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","[Madde 17 – Herkes, yaşama, maddi ve manevi va...","[chunk_000373, chunk_000366, chunk_000275]","[Türkiye Cumhuriyeti Anayasası, Türkiye Cumhur..."
3,3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,[31/7/2008-5797/10 md.) Bu fıkrada düzenlenen ...,"[chunk_003745, chunk_003712, chunk_000386]","[Türkiye Cumhuriyeti İş Kanunu, Türkiye Cumhur..."
4,4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,[(5) Bu suçun ihmali davranışla işlenmesi hâli...,"[chunk_003288, chunk_003274, chunk_003278]","[Türk Ceza Kanunu, Türk Ceza Kanunu, Türk Ceza..."


In [15]:
retrieval_eval_save_df = retrieval_eval_df.copy()

retrieval_eval_save_df["retrieved_contexts"] = retrieval_eval_save_df["retrieved_contexts"].apply(
    lambda x: json.dumps(x, ensure_ascii=False)
)

retrieval_eval_save_df["retrieved_chunk_ids"] = retrieval_eval_save_df["retrieved_chunk_ids"].apply(
    lambda x: "; ".join(x)
)

retrieval_eval_save_df["retrieved_sources"] = retrieval_eval_save_df["retrieved_sources"].apply(
    lambda x: "; ".join(x)
)

retrieval_contexts_path = f"{metrics_path}/llm_eval_retrieved_contexts_20.csv"

retrieval_eval_save_df.to_csv(
    retrieval_contexts_path,
    index=False,
    encoding="utf-8-sig"
)

print("Retrieved contexts saved:", retrieval_contexts_path)

Retrieved contexts saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/llm_eval_retrieved_contexts_20.csv


In [16]:
del embedding_model
del reranker

gc.collect()
torch.cuda.empty_cache()

print("Retrieval models deleted.")

Retrieval models deleted.


In [18]:
SYSTEM_INSTRUCTION = """Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.
Cevabı yalnızca verilen bağlama göre üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, net ve Türkçe olmalı."""


def build_eval_prompt(question, contexts):
    context_text = "\n\n".join([
        f"[Bağlam {i+1}]\n{ctx}"
        for i, ctx in enumerate(contexts)
    ])

    user_prompt = f"""Bağlam:
{context_text}

Soru:
{question}

Cevap:"""

    prompt = f"""<s>[INST] {SYSTEM_INSTRUCTION}

{user_prompt} [/INST]"""

    return prompt

In [19]:
def generate_answer_only(model, tokenizer, prompt, max_new_tokens=180):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()


def clean_answer(text):
    text = str(text).strip()

    # Model bazen gereksiz prefix bırakırsa temizleyelim
    for marker in ["Cevap:", "Yanıt:", "Answer:"]:
        if marker in text:
            text = text.split(marker)[-1].strip()

    return text

**Base Mistral RAG generation**

In [20]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

base_model.config.use_cache = False

print("Base model loaded.")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Base model loaded.


In [21]:
base_results = []

for i, row in tqdm(retrieval_eval_df.iterrows(), total=len(retrieval_eval_df)):
    prompt = build_eval_prompt(
        question=row["question"],
        contexts=row["retrieved_contexts"]
    )

    generated = generate_answer_only(
        model=base_model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=180
    )

    base_results.append(clean_answer(generated))

retrieval_eval_df["base_generated_answer"] = base_results

display(retrieval_eval_df[[
    "question",
    "expected_answer",
    "base_generated_answer"
]].head())

  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 20/20 [04:05<00:00, 12.29s/it]


,question,expected_answer,base_generated_answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,Anayasanın 101. Madde'si hakkında tartışılan k...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Madde, herkesin dil, ırk, cinsi..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Anayasanın 17. Madde, herkesin yaşama hakkını ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Madde 1'deki işe iade talebi sürecindeki süre ...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"İşkence fiilleri, insanların duyularından veya..."


In [22]:
del base_model

gc.collect()
torch.cuda.empty_cache()

print("Base model deleted.")

Base model deleted.


**Fine-tuned QLoRA RAG generation**

In [23]:
print("Adapter path:", adapter_path)
print("Adapter model exists:", os.path.exists(f"{adapter_path}/adapter_model.safetensors"))
print("Adapter config exists:", os.path.exists(f"{adapter_path}/adapter_config.json"))

Adapter path: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps
Adapter model exists: True
Adapter config exists: True


In [24]:
ft_base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

ft_base_model.config.use_cache = False

ft_model = PeftModel.from_pretrained(
    ft_base_model,
    adapter_path
)

ft_model.eval()

print("Fine-tuned adapter loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Fine-tuned adapter loaded.


In [25]:
finetuned_results = []

for i, row in tqdm(retrieval_eval_df.iterrows(), total=len(retrieval_eval_df)):
    prompt = build_eval_prompt(
        question=row["question"],
        contexts=row["retrieved_contexts"]
    )

    generated = generate_answer_only(
        model=ft_model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=180
    )

    finetuned_results.append(clean_answer(generated))

retrieval_eval_df["finetuned_generated_answer"] = finetuned_results

display(retrieval_eval_df[[
    "question",
    "expected_answer",
    "base_generated_answer",
    "finetuned_generated_answer"
]].head())

  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 20/20 [09:41<00:00, 29.05s/it]


,question,expected_answer,base_generated_answer,finetuned_generated_answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,Anayasanın 101. Madde'si hakkında tartışılan k...,Anayasanın 101. maddesi ile ilgili tartışmalar...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Madde, herkesin dil, ırk, cinsi...","Anayasanın 10. Madde, herkes için kanun önünde..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Anayasanın 17. Madde, herkesin yaşama hakkını ...",Yaşama hakkının sını
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Madde 1'deki işe iade talebi sürecindeki süre ...,"İş sözleşmesi feshedilen işçi, fesih bildirimi..."
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"İşkence fiilleri, insanların duyularından veya...","üşürtülmesine,\n\n[Bağlam 4]\nf) Yaşamını tehl..."


In [26]:
del ft_model
del ft_base_model

gc.collect()
torch.cuda.empty_cache()

print("Fine-tuned model deleted.")

Fine-tuned model deleted.


In [27]:
llm_eval_results_df = retrieval_eval_df.copy()

llm_eval_results_df["retrieved_contexts"] = llm_eval_results_df["retrieved_contexts"].apply(
    lambda x: json.dumps(x, ensure_ascii=False)
)

llm_eval_results_df["retrieved_chunk_ids"] = llm_eval_results_df["retrieved_chunk_ids"].apply(
    lambda x: "; ".join(x)
)

llm_eval_results_df["retrieved_sources"] = llm_eval_results_df["retrieved_sources"].apply(
    lambda x: "; ".join(x)
)

results_path = f"{metrics_path}/base_vs_finetuned_llm_rag_generation_results_20.csv"

llm_eval_results_df.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig"
)

print("LLM evaluation results saved:", results_path)
display(llm_eval_results_df.head())

LLM evaluation results saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/base_vs_finetuned_llm_rag_generation_results_20.csv


,index,question,expected_answer,retrieved_contexts,retrieved_chunk_ids,retrieved_sources,base_generated_answer,finetuned_generated_answer
0,0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"[""Cumhurbaşkanlığı seçiminde birinci oylamada ...",chunk_000537; chunk_000277; chunk_000366,Türkiye Cumhuriyeti Anayasası; Türkiye Cumhuri...,Anayasanın 101. Madde'si hakkında tartışılan k...,Anayasanın 101. maddesi ile ilgili tartışmalar...
1,1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","[""Madde 10 – Herkes, dil, ırk, renk, cinsiyet,...",chunk_000274; chunk_000277; chunk_000366,Türkiye Cumhuriyeti Anayasası; Türkiye Cumhuri...,"Anayasanın 10. Madde, herkesin dil, ırk, cinsi...","Anayasanın 10. Madde, herkes için kanun önünde..."
2,2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","[""Madde 17 – Herkes, yaşama, maddi ve manevi v...",chunk_000373; chunk_000366; chunk_000275,Türkiye Cumhuriyeti Anayasası; Türkiye Cumhuri...,"Anayasanın 17. Madde, herkesin yaşama hakkını ...",Yaşama hakkının sını
3,3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,"[""31/7/2008-5797/10 md.) Bu fıkrada düzenlenen...",chunk_003745; chunk_003712; chunk_000386,Türkiye Cumhuriyeti İş Kanunu; Türkiye Cumhuri...,Madde 1'deki işe iade talebi sürecindeki süre ...,"İş sözleşmesi feshedilen işçi, fesih bildirimi..."
4,4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"[""(5) Bu suçun ihmali davranışla işlenmesi hâl...",chunk_003288; chunk_003274; chunk_003278,Türk Ceza Kanunu; Türk Ceza Kanunu; Türk Ceza ...,"İşkence fiilleri, insanların duyularından veya...","üşürtülmesine,\n\n[Bağlam 4]\nf) Yaşamını tehl..."


In [28]:
manual_scoring_df = llm_eval_results_df.copy()

manual_scoring_df["base_manual_score"] = ""
manual_scoring_df["finetuned_manual_score"] = ""
manual_scoring_df["base_error_type"] = ""
manual_scoring_df["finetuned_error_type"] = ""
manual_scoring_df["notes"] = ""

manual_scoring_path = f"{metrics_path}/base_vs_finetuned_llm_rag_manual_scoring_template_20.csv"

manual_scoring_df.to_csv(
    manual_scoring_path,
    index=False,
    encoding="utf-8-sig"
)

print("Manual scoring template saved:", manual_scoring_path)

Manual scoring template saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/base_vs_finetuned_llm_rag_manual_scoring_template_20.csv


In [29]:
for i, row in llm_eval_results_df.head(5).iterrows():
    print("=" * 120)
    print("INDEX:", row["index"])
    print("QUESTION:")
    print(row["question"])

    print("\nEXPECTED:")
    print(row["expected_answer"])

    print("\nBASE GENERATED:")
    print(row["base_generated_answer"])

    print("\nFINE-TUNED GENERATED:")
    print(row["finetuned_generated_answer"])

    print("\nRETRIEVED CHUNKS:")
    print(row["retrieved_chunk_ids"])

INDEX: 0
QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

BASE GENERATED:
Anayasanın 101. Madde'si hakkında tartışılan konular, genellikle Cumhurbaşkanlığı seçimlerinde gerekli çoğunluğun sağlanamaması durumunda ikinci oylama konusu tutulur. Bu tartışmalar, ikinci oylama hakkında zaman, mali, teknik ve diğer aspectleri ile ilgili olabilir. Ancak, Anayasanın 101. Madde'si metninde açıkça belirtilmemiş olduğu gibi, ikinci oylama hakkının tamam

FINE-TUNED GENERATED:
Anayasanın 101. maddesi ile ilgili tartışmalar, genel olarak, Türkiye Cumhuriyetinin Anayasasında yapılan içtimaınin ve çalışmalarının nasıl yapılacağı ve nasıl yapılmış olduğu gibi, Cumhurbaşkanlığı seçiminde gerekli çoğunluğun sağlanamaması halinde yapılacak oylamaların nasıl yapılacağı ile ilgili olur.

Bu maddede açıkça bulunmayan

RETRIEVED CHUNKS:
chunk_000537; chunk_000277; chunk_000366
INDEX: 1
QUES

In [30]:
llm_eval_summary_df = pd.DataFrame([{
    "base_model": model_name,
    "fine_tuned_adapter": adapter_path,
    "eval_sample_size": len(llm_eval_results_df),
    "retrieval_setup": "source-aware + article-aware + Turkish BGE reranker, context top-3",
    "generation_max_new_tokens": 180,
    "results_path": results_path,
    "manual_scoring_template_path": manual_scoring_path
}])

summary_path = f"{metrics_path}/base_vs_finetuned_llm_rag_eval_summary_20.csv"

llm_eval_summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

display(llm_eval_summary_df)

print("Evaluation summary saved:", summary_path)

,base_model,fine_tuned_adapter,eval_sample_size,retrieval_setup,generation_max_new_tokens,results_path,manual_scoring_template_path
0,mistralai/Mistral-7B-Instruct-v0.2,/content/drive/MyDrive/turkish_legal_rag/outpu...,20,source-aware + article-aware + Turkish BGE rer...,180,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...


Evaluation summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/base_vs_finetuned_llm_rag_eval_summary_20.csv


In [31]:
print("FINAL CHECK")
print("=" * 80)

files_to_check = [
    results_path,
    manual_scoring_path,
    summary_path,
    retrieval_contexts_path
]

for file in files_to_check:
    print(file)
    print("Exists:", os.path.exists(file))
    if os.path.exists(file):
        print("Size KB:", round(os.path.getsize(file) / 1024, 2))
    print("-" * 80)

print("Done.")

FINAL CHECK
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/base_vs_finetuned_llm_rag_generation_results_20.csv
Exists: True
Size KB: 45.43
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/base_vs_finetuned_llm_rag_manual_scoring_template_20.csv
Exists: True
Size KB: 45.61
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/base_vs_finetuned_llm_rag_eval_summary_20.csv
Exists: True
Size KB: 0.55
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/llm_eval_retrieved_contexts_20.csv
Exists: True
Size KB: 32.95
--------------------------------------------------------------------------------
Done.


In [32]:
import numpy as np
import pandas as pd

scored_df = llm_eval_results_df.copy()

base_manual_scores = [
    0.0,  # 0
    0.5,  # 1
    0.5,  # 2
    0.0,  # 3
    0.0,  # 4
    0.5,  # 5
    0.0,  # 6
    0.0,  # 7
    0.0,  # 8
    0.0,  # 9
    0.5,  # 10
    0.0,  # 11
    0.0,  # 12
    1.0,  # 13
    0.5,  # 14
    0.5,  # 15
    1.0,  # 16
    0.0,  # 17
    1.0,  # 18
    np.nan # 19 invalid
]

finetuned_manual_scores = [
    0.0,  # 0
    0.5,  # 1
    0.0,  # 2
    0.0,  # 3
    0.0,  # 4
    0.0,  # 5
    0.0,  # 6
    0.5,  # 7
    0.0,  # 8
    0.0,  # 9
    0.5,  # 10
    0.0,  # 11
    0.5,  # 12
    0.5,  # 13
    0.5,  # 14
    0.5,  # 15
    1.0,  # 16
    0.0,  # 17
    0.5,  # 18
    np.nan # 19 invalid
]

base_error_types = [
    "wrong_generation",
    "partial_answer",
    "partial_answer",
    "wrong_context",
    "wrong_context",
    "partial_answer",
    "wrong_context",
    "wrong_generation",
    "wrong_generation",
    "wrong_generation",
    "partial_answer",
    "wrong_context",
    "wrong_generation",
    "correct",
    "partial_answer",
    "partial_answer",
    "correct",
    "wrong_generation",
    "correct",
    "invalid_sample"
]

finetuned_error_types = [
    "wrong_generation",
    "partial_answer",
    "incomplete_answer",
    "wrong_context",
    "context_copy",
    "wrong_generation",
    "wrong_generation",
    "partial_answer_prompt_leakage",
    "wrong_generation",
    "context_copy",
    "partial_answer_prompt_leakage",
    "wrong_context",
    "partial_answer",
    "partial_answer_context_drift",
    "partial_answer",
    "partial_answer",
    "correct",
    "wrong_generation",
    "partial_answer",
    "invalid_sample"
]

notes = [
    "Both models fail to focus on presidential eligibility discussion.",
    "Both models capture equality principle partially but outputs are linguistically weak.",
    "Base partially captures Article 17, fine-tuned answer is incomplete.",
    "Retrieved context is mostly wrong; date answer is not produced.",
    "Both models fail; fine-tuned copies irrelevant legal text.",
    "Base partially mentions oath obligation, fine-tuned is mostly wrong.",
    "Article 122 answer is not recovered.",
    "Fine-tuned states seven days but includes prompt leakage.",
    "Both models fail to clearly answer that it is unconstitutional.",
    "Both models fail to mention armed forces exclusion from DDK inspection.",
    "Both models partially capture arbitrary restriction issue.",
    "Expected KVKK definition is not supported by retrieved contexts.",
    "Fine-tuned captures that Constitutional Court decisions are final.",
    "Base is correct; fine-tuned starts correctly but drifts.",
    "Both partially capture nationalization/privatization principles.",
    "Both partially answer personal data processing conditions.",
    "Both answer Article 140 well.",
    "Both fail to identify the expected article.",
    "Base gives stronger answer about appeal to the Board.",
    "Invalid sample: question and expected answer are mismatched."
]

scored_df["is_valid_sample"] = True
scored_df.loc[19, "is_valid_sample"] = False

scored_df["base_manual_score"] = base_manual_scores
scored_df["finetuned_manual_score"] = finetuned_manual_scores
scored_df["base_error_type"] = base_error_types
scored_df["finetuned_error_type"] = finetuned_error_types
scored_df["notes"] = notes

valid_scored_df = scored_df[scored_df["is_valid_sample"] == True].copy()

base_manual_accuracy = valid_scored_df["base_manual_score"].mean()
finetuned_manual_accuracy = valid_scored_df["finetuned_manual_score"].mean()

print("Valid sample count:", len(valid_scored_df))
print("Base Mistral RAG manual accuracy:", base_manual_accuracy)
print("Fine-tuned Mistral QLoRA RAG manual accuracy:", finetuned_manual_accuracy)

llm_manual_score_summary_df = pd.DataFrame([
    {
        "method": "Base Mistral RAG",
        "manual_accuracy": base_manual_accuracy,
        "valid_sample_count": len(valid_scored_df)
    },
    {
        "method": "Fine-tuned Mistral QLoRA RAG",
        "manual_accuracy": finetuned_manual_accuracy,
        "valid_sample_count": len(valid_scored_df)
    }
])

display(llm_manual_score_summary_df)

Valid sample count: 19
Base Mistral RAG manual accuracy: 0.3157894736842105
Fine-tuned Mistral QLoRA RAG manual accuracy: 0.2631578947368421


,method,manual_accuracy,valid_sample_count
0,Base Mistral RAG,0.315789,19
1,Fine-tuned Mistral QLoRA RAG,0.263158,19


In [33]:
scored_path = f"{metrics_path}/base_vs_finetuned_llm_rag_manual_scored_20.csv"
summary_path = f"{metrics_path}/base_vs_finetuned_llm_rag_manual_score_summary_20.csv"

scored_df.to_csv(
    scored_path,
    index=False,
    encoding="utf-8-sig"
)

llm_manual_score_summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Manual scored results saved:", scored_path)
print("Manual score summary saved:", summary_path)

Manual scored results saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/base_vs_finetuned_llm_rag_manual_scored_20.csv
Manual score summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/base_vs_finetuned_llm_rag_manual_score_summary_20.csv
